## Orchestrated Multi Task Execution

# The Orchestration Strategy

In the previous lesson, we spent our time planning. We built the "brain" of our app using a Product Requirements Document (PRD) and a Technical Specification. Now, it is time to build the actual application.

Building a production-ready app involves dozens of small tasks. If you did this manually, you would spend hours writing boilerplate code, setting up databases, and fixing small bugs. Instead, we use an **Orchestration Strategy**.

Orchestration means you act as the manager. You will oversee a Main Session that coordinates 23 different tasks across 6 development phases. To do this, you will delegate work to specialized agents:

* **task-executor:** This is your primary builder. It writes the code and the initial tests.
* **test-enhancer:** This agent reviews existing tests and adds edge cases to ensure the app doesn't break.
* **recipe-validator:** This agent checks if the complex business logic (such as ingredient scaling) matches our specific rules.
* **doc-updater:** This keeps your documentation, such as the API map (OpenAPI), in sync with your code.

In this lesson, you will learn how to guide these agents to build the RecipeBox app.

---

## Recall: The SDD Foundation

Before we start the engines, let’s remember how Spec-Driven Development (SDD) works. Everything the agents build is based on the files we created earlier:

1. **The Specification:** This is the source of truth. If the agent needs to know what fields a Recipe has, it looks at `@specs/recipebox/specification.md`.
2. **The Constitution:** This is a file (often named `CLAUDE.md`) that contains your "rules of the house." It tells the agents to use specific styles, such as using type hints in Python or following certain security patterns.
3. **The Task List:** We have a clear list of tasks (`T001` through `T023`). We never guess what to do next; we simply follow the list.

By keeping these specs in the agent's context, we ensure that the code stays consistent even as the project grows.

---

## The Foundation Phase: Execute, Validate, Commit

The first phase of our build is the Foundation. This involves creating the database models. We use an **Atomic Loop** for every task: **Delegate ➔ Validate ➔ Commit**.

Let’s look at how you would handle Task `T001`: Create the Recipe Model. First, you give the `task-executor` a clear instruction. You tell it exactly what to build and which spec to follow.

```text
Task(task-executor): "Execute T001 from @specs/recipebox/tasks.md: Create Recipe model with fields (name, description, prep_time, cook_time, servings), relationships (one-to-many RecipeIngredients), and unit tests."

```

The agent will then create the files. Once it has finished, it provides a report. Your job is not to rewrite the code, but to validate it quickly. You can do this by running a small script to ensure the model actually exists and loads correctly.

First, let's see how we import our new model:

```python
from src.recipebox.models.recipe import Recipe

# We check if we can create a simple instance of the model
new_recipe = Recipe(name="Pasta", servings=4)
print(f"Model created: {new_recipe.name}")

```

In this snippet, we are verifying that the `Recipe` class was actually created in the expected folder. Next, we check if the fields we requested are available.

```python
from src.recipebox.models.recipe import Recipe

new_recipe = Recipe(
    name="Pasta", 
    description="Delicious Italian pasta", 
    servings=4
)
print(f"Recipe: {new_recipe.name}")
print(f"Servings: {new_recipe.servings}")

```

By running this, you confirm that the agent followed the Recipe definition from your spec. The output should look like this:

```text
Recipe: Pasta
Servings: 4

```

Once you are satisfied, you perform an **Atomic Commit**. This means you save only the files related to this specific task to your version control (Git). This keeps your project history clean and easy to track.

---

## Parallel Workflows: Recipe API and Meal Planning

One of the biggest benefits of orchestration is the ability to do two things at once. In a traditional workflow, you might build the Recipe API, finish it, and then start the Meal Planning feature. With orchestration, because we have a solid foundation, these two features can be built in parallel tracks.

* **Track A:** Focuses on the Recipe API (CRUD operations such as creating, reading, updating, and deleting recipes).
* **Track B:** Focuses on the Meal Planning logic (scheduling recipes for certain dates).

Since both tracks use the Foundation models we built in Phase 1, they do not interfere with each other. You can have one agent session working on Track A while you (or another agent session) work on Track B. In CodeSignal, run Track A and Track B sequentially in one session, or use two sessions/tabs to simulate parallelism; coordinate by committing Foundation first so both tracks see the same base.

When both tracks are finished, you perform an **Integration Checkpoint**. This is where you ensure that the two features interact correctly. For example, you might try to add a recipe (from Track A) to a meal plan (from Track B).

```python
# Integration test example (pytest + FastAPI TestClient)
from fastapi.testclient import TestClient
from src.recipebox.main import app

client = TestClient(app)

def test_add_recipe_to_meal_plan():
    # Create recipe and meal plan via API
    r_recipe = client.post("/api/recipes/", json={"name": "Pasta", "servings": 4})
    assert r_recipe.status_code == 201
    recipe_id = r_recipe.json()["id"]
    
    r_plan = client.post("/api/meal-plans/", json={"week_start_date": "2025-03-17"})
    assert r_plan.status_code == 201
    plan_id = r_plan.json()["id"]
    
    # Add recipe to plan
    r_add = client.post(f"/api/meal-plans/{plan_id}/meals",
        json={"recipe_id": recipe_id, "date": "2025-03-17", "meal_type": "dinner", "servings": 4})
    assert r_add.status_code == 201
    assert r_add.json()["recipe_id"] == recipe_id

```

By working this way, you can cut the total "calendar time" it takes to build the app by nearly 50%.

---

## The Automated Quality Pipeline

Once the features are built, we do not simply hope they work; we use a Quality Pipeline. This is Phase 6 of our plan. Instead of writing every single test case yourself, you delegate this to the `test-enhancer`. This agent looks at your code and identifies "holes" in your testing.

For example, if you have a service that scales recipe ingredients, the `test-enhancer` might suggest testing what happens if a user enters 0 servings.

```text
Task(test-enhancer): "Enhance RecipeService test coverage. Focus on edge cases like 0 servings or very large numbers."

```

After the agent runs, you check the coverage report. In a production app, we usually aim for 95% coverage or higher. This means 95% of your code lines are exercised by a test.

Finally, we use the `doc-updater`. In this course we use automated API documentation: the agent reads your code and updates OpenAPI/Swagger so other developers know how to use your API. Many teams also hand-author or curate OpenAPI specs in API-first workflows; here we emphasize the automated pattern.

---

## Measuring Success and Performance

A professional developer doesn't just build; they measure. To prove that orchestration is superior to manual coding, we track our metrics. You should keep a log of two things:

1. **Agent Execution Time:** How long the AI took to write the code.
2. **Your Validation Time:** How long you took to review and approve it.

Look at this comparison for a typical foundation phase:

| Phase | Tasks | Agent Time | Your Validation | Total | Est. Manual Time |
| --- | --- | --- | --- | --- | --- |
| **Foundation** | 6 | 24 min | 18 min | 42 min | 200 min |

In this example, the orchestrated approach took about 42 minutes, while performing the tasks manually would have taken over 3 hours. This represents an **efficiency gain of 79%**. By keeping these metrics, you can show your team or your clients exactly how much faster you are delivering high-quality code.

---

## Lesson Summary and Practice Prep

In this lesson, you learned the Orchestration Strategy for building a production app. We moved from the planning phase into implementation by delegating 23 tasks to specialized agents.

Here is what we covered:

* **The Atomic Loop:** How to Delegate, Validate, and Commit code for every task.
* **Parallel Tracks:** Using orchestration to build multiple features (Recipe API and Meal Planning) simultaneously.
* **Quality Automation:** Using the `test-enhancer` and `doc-updater` to ensure the app is robust and well-documented.
* **Efficiency Metrics:** Tracking time to prove the speed and quality gains of this methodology.

In the upcoming practice exercises, you will enter the CodeSignal IDE and begin Phase 1: Foundation. You will be responsible for starting the orchestration sessions and validating the models created by the agents. Get ready to build!

## Initialize RecipeBox Foundation Models

Now that you understand the orchestration strategy, it's time to apply it by building the first part of the RecipeBox data layer. You will implement four foundation models (not the full seven yet) so you can practice relationships and cascade without repeating the same work in the next task.

Your job is to complete these four model classes with the necessary fields and relationships:

    Recipe: Stores recipe information (name, description, cooking times, and servings) and links to the user who created it.
    Ingredient: Represents individual ingredients with name, category, and description.
    RecipeIngredient: A join table that connects recipes to ingredients with specific amounts and units (use Numeric(10, 2) for amount, not Float).
    MealPlan: Organizes meals for a user by week (foundation for the next task).

Add proper relationships and cascade deletes (e.g. removing a recipe removes its recipe_ingredients). Write unit tests to verify fields, relationships, and cascade behavior for these four models.

Deliverables: The four model files, __init__.py exporting them, and unit tests for these models. Task 2 will add the remaining models (MealPlanItem, ShoppingList, ShoppingListItem), migration, and a first route or orchestration step.

```
# init.py
# TODO: Import all models once they are created


# recipe.py
from sqlalchemy import Column, String, Integer, Text, ForeignKey, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from src.recipebox.database import Base


class Recipe(Base):
    __tablename__ = "recipes"

    id = Column(Integer, primary_key=True, index=True)
    # TODO: Add name field (String 255, not null)
    # TODO: Add description field (Text, optional)
    # TODO: Add prep_time field (Integer, minutes)
    # TODO: Add cook_time field (Integer, minutes)
    # TODO: Add servings field (Integer, not null)
    # TODO: Add created_by field (String, not null)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # TODO: Add relationship to RecipeIngredient with cascade delete

    def __repr__(self):
        return f"<Recipe(id={self.id}, name={self.name}, servings={self.servings})>"

```

```
# TODO Items Collection

**Generated:** 2026-07-15
**Total TODO items found:** 50

---

## Models

### src/recipebox/models/__init__.py

```python
# TODO: Import all models once they are created
```

---

### src/recipebox/models/ingredient.py

```python
from sqlalchemy import Column, String, Integer, Text
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class Ingredient(Base):
    __tablename__ = "ingredients"

    id = Column(Integer, primary_key=True, index=True)
    # TODO: Add name field (String 255, not null, unique)
    # TODO: Add category field (String 100, optional)
    # TODO: Add description field (Text, optional)

    # TODO: Add relationship to RecipeIngredient

    def __repr__(self):
        return f"<Ingredient(id={self.id}, name={self.name}, category={self.category})>"
```

**TODOs:**
- Add name field (String 255, not null, unique)
- Add category field (String 100, optional)
- Add description field (Text, optional)
- Add relationship to RecipeIngredient

---

### src/recipebox/models/recipe.py

```python
from sqlalchemy import Column, String, Integer, Text, ForeignKey, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from src.recipebox.database import Base


class Recipe(Base):
    __tablename__ = "recipes"

    id = Column(Integer, primary_key=True, index=True)
    # TODO: Add name field (String 255, not null)
    # TODO: Add description field (Text, optional)
    # TODO: Add prep_time field (Integer, minutes)
    # TODO: Add cook_time field (Integer, minutes)
    # TODO: Add servings field (Integer, not null)
    # TODO: Add created_by field (String, not null)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # TODO: Add relationship to RecipeIngredient with cascade delete

    def __repr__(self):
        return f"<Recipe(id={self.id}, name={self.name}, servings={self.servings})>"
```

**TODOs:**
- Add name field (String 255, not null)
- Add description field (Text, optional)
- Add prep_time field (Integer, minutes)
- Add cook_time field (Integer, minutes)
- Add servings field (Integer, not null)
- Add created_by field (String, not null)
- Add relationship to RecipeIngredient with cascade delete

---

### src/recipebox/models/recipe_ingredient.py

```python
from sqlalchemy import Column, String, Integer, Numeric, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class RecipeIngredient(Base):
    __tablename__ = "recipe_ingredients"

    id = Column(Integer, primary_key=True, index=True)
    # TODO: Add recipe_id field (Integer, ForeignKey to recipes.id, not null)
    # TODO: Add ingredient_id field (Integer, ForeignKey to ingredients.id, not null)
    # TODO: Add amount field (Float, not null)
    # TODO: Add unit field (String 50, not null)

    # TODO: Add relationship to Recipe (back_populates="recipe_ingredients")
    # TODO: Add relationship to Ingredient (back_populates="recipe_ingredients")

    def __repr__(self):
        return f"<RecipeIngredient(recipe_id={self.recipe_id}, ingredient_id={self.ingredient_id}, amount={self.amount} {self.unit})>"
```

**TODOs:**
- Add recipe_id field (Integer, ForeignKey to recipes.id, not null)
- Add ingredient_id field (Integer, ForeignKey to ingredients.id, not null)
- Add amount field (Float, not null)
- Add unit field (String 50, not null)
- Add relationship to Recipe (back_populates="recipe_ingredients")
- Add relationship to Ingredient (back_populates="recipe_ingredients")

---

### src/recipebox/models/meal_plan.py

```python
from sqlalchemy import Column, String, Integer, Date, ForeignKey, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from src.recipebox.database import Base


class MealPlan(Base):
    __tablename__ = "meal_plans"

    id = Column(Integer, primary_key=True, index=True)
    # TODO: Add user_id field (String, not null)
    # TODO: Add week_start_date field (Date, not null)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # TODO: Add relationship to MealPlanItem with cascade delete

    def __repr__(self):
        return f"<MealPlan(id={self.id}, user_id={self.user_id}, week_start={self.week_start_date})>"
```

**TODOs:**
- Add user_id field (String, not null)
- Add week_start_date field (Date, not null)
- Add relationship to MealPlanItem with cascade delete

---

## Tests

### tests/unit/test_ingredient_model.py

```python
from src.recipebox.models.ingredient import Ingredient


def test_ingredient_fields(db_session):
    """Test Ingredient model has correct fields."""
    # TODO: Create an Ingredient with name, category, description
    # TODO: Add to session and commit
    # TODO: Assert all fields are set correctly
    pass


def test_ingredient_unique_name(db_session):
    """Test Ingredient name must be unique."""
    # TODO: Create two Ingredients with the same name
    # TODO: Try to add both to database
    # TODO: Assert second one raises an IntegrityError
    pass


def test_ingredient_str_repr():
    """Test Ingredient string representation."""
    # TODO: Create an Ingredient
    # TODO: Assert repr contains ingredient name and category
    pass
```

**TODOs:**
- Create an Ingredient with name, category, description
- Add to session and commit
- Assert all fields are set correctly
- Create two Ingredients with the same name
- Try to add both to database
- Assert second one raises an IntegrityError
- Create an Ingredient
- Assert repr contains ingredient name and category

---

### tests/unit/test_recipe_model.py

```python
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.recipe_ingredient import RecipeIngredient
from src.recipebox.models.ingredient import Ingredient


def test_recipe_fields(db_session):
    """Test Recipe model has correct fields."""
    # TODO: Create a Recipe with name, description, prep_time, cook_time, servings, created_by
    # TODO: Add to session and commit
    # TODO: Assert all fields are set correctly
    pass


def test_recipe_ingredients_relationship(db_session):
    """Test Recipe has relationship with RecipeIngredients."""
    # TODO: Create Recipe, Ingredient, and RecipeIngredient
    # TODO: Add all to session and commit
    # TODO: Assert recipe.recipe_ingredients contains the RecipeIngredient
    # TODO: Assert the RecipeIngredient has correct amount and unit
    pass


def test_recipe_cascade_delete(db_session):
    """Test deleting Recipe cascades to RecipeIngredients."""
    # TODO: Create Recipe with RecipeIngredient
    # TODO: Delete the Recipe
    # TODO: Assert RecipeIngredient is also deleted
    pass


def test_recipe_str_repr():
    """Test Recipe string representation."""
    # TODO: Create a Recipe
    # TODO: Assert repr contains recipe name and servings
    pass
```

**TODOs:**
- Create a Recipe with name, description, prep_time, cook_time, servings, created_by
- Add to session and commit
- Assert all fields are set correctly
- Create Recipe, Ingredient, and RecipeIngredient
- Add all to session and commit
- Assert recipe.recipe_ingredients contains the RecipeIngredient
- Assert the RecipeIngredient has correct amount and unit
- Create Recipe with RecipeIngredient
- Delete the Recipe
- Assert RecipeIngredient is also deleted
- Create a Recipe
- Assert repr contains recipe name and servings

---

### tests/unit/test_recipe_ingredient_model.py

```python
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.ingredient import Ingredient
from src.recipebox.models.recipe_ingredient import RecipeIngredient


def test_recipe_ingredient_fields(db_session):
    """Test RecipeIngredient model has correct fields."""
    # TODO: Create Recipe, Ingredient, and RecipeIngredient
    # TODO: Add to session and commit
    # TODO: Assert all fields are set correctly
    pass


def test_recipe_ingredient_relationships(db_session):
    """Test RecipeIngredient relationships work both ways."""
    # TODO: Create Recipe, Ingredient, and RecipeIngredient
    # TODO: Assert recipe.recipe_ingredients[0].ingredient.name is correct
    # TODO: Assert ingredient.recipe_ingredients[0].recipe.name is correct
    pass


def test_recipe_ingredient_str_repr():
    """Test RecipeIngredient string representation."""
    # TODO: Create RecipeIngredient
    # TODO: Assert repr contains amount and unit
    pass
```

**TODOs:**
- Create Recipe, Ingredient, and RecipeIngredient
- Add to session and commit
- Assert all fields are set correctly
- Create Recipe, Ingredient, and RecipeIngredient
- Assert recipe.recipe_ingredients[0].ingredient.name is correct
- Assert ingredient.recipe_ingredients[0].recipe.name is correct
- Create RecipeIngredient
- Assert repr contains amount and unit

---

### tests/unit/test_meal_plan_model.py

```python
from datetime import date
from src.recipebox.models.meal_plan import MealPlan


def test_meal_plan_fields(db_session):
    """Test MealPlan model has correct fields."""
    # TODO: Create a MealPlan with user_id and week_start_date
    # TODO: Add to session and commit
    # TODO: Assert all fields are set correctly
    pass


def test_meal_plan_str_repr():
    """Test MealPlan string representation."""
    # TODO: Create a MealPlan
    # TODO: Assert repr contains user_id and week_start_date
    pass
```

**TODOs:**
- Create a MealPlan with user_id and week_start_date
- Add to session and commit
- Assert all fields are set correctly
- Create a MealPlan
- Assert repr contains user_id and week_start_date

---

## Summary by Category

### Model Implementation (20 items)
- **Ingredient Model:** 4 TODOs
- **Recipe Model:** 7 TODOs
- **RecipeIngredient Model:** 6 TODOs
- **MealPlan Model:** 3 TODOs

### Test Implementation (30 items)
- **Ingredient Tests:** 8 TODOs
- **Recipe Tests:** 12 TODOs
- **RecipeIngredient Tests:** 7 TODOs
- **MealPlan Tests:** 5 TODOs

### Module Organization (1 item)
- **Model Imports:** 1 TODO

```

Here is the complete, placeholder-free, production-ready code implementation for all four foundational models, their module organizational entry point, and the accompanying comprehensive unit test suite to verify fields, data types, relationships, and cascade behaviors.

---

### 📋 1. Core Model Layers & Module Initialization

#### Model Export Hub (`src/recipebox/models/__init__.py`)

```python
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.ingredient import Ingredient
from src.recipebox.models.recipe_ingredient import RecipeIngredient
from src.recipebox.models.meal_plan import MealPlan

__all__ = ["Recipe", "Ingredient", "RecipeIngredient", "MealPlan"]

```

#### Ingredient Model Definition (`src/recipebox/models/ingredient.py`)

```python
from sqlalchemy import Column, String, Integer, Text
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class Ingredient(Base):
    __tablename__ = "ingredients"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String(255), nullable=False, unique=True)
    category = Column(String(100), nullable=True)
    description = Column(Text, nullable=True)

    # Relationship to RecipeIngredient back-populating bidirectional reference
    recipe_ingredients = relationship(
        "RecipeIngredient", 
        back_populates="ingredient", 
        cascade="all, delete-orphan"
    )

    def __repr__(self):
        return f"<Ingredient(id={self.id}, name='{self.name}', category='{self.category}')>"

```

#### Recipe Model Definition (`src/recipebox/models/recipe.py`)

```python
from sqlalchemy import Column, String, Integer, Text, ForeignKey, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from src.recipebox.database import Base


class Recipe(Base):
    __tablename__ = "recipes"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String(255), nullable=False)
    description = Column(Text, nullable=True)
    prep_time = Column(Integer, nullable=True)  # Minutes
    cook_time = Column(Integer, nullable=True)  # Minutes
    servings = Column(Integer, nullable=False)
    created_by = Column(String(255), nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relationship to RecipeIngredient with complete cascade delete capability
    recipe_ingredients = relationship(
        "RecipeIngredient", 
        back_populates="recipe", 
        cascade="all, delete-orphan"
    )

    def __repr__(self):
        return f"<Recipe(id={self.id}, name='{self.name}', servings={self.servings})>"

```

#### RecipeIngredient Junction Model Definition (`src/recipebox/models/recipe_ingredient.py`)

```python
from sqlalchemy import Column, String, Integer, Numeric, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class RecipeIngredient(Base):
    __tablename__ = "recipe_ingredients"

    id = Column(Integer, primary_key=True, index=True)
    recipe_id = Column(Integer, ForeignKey("recipes.id", ondelete="CASCADE"), nullable=False)
    ingredient_id = Column(Integer, ForeignKey("ingredients.id", ondelete="CASCADE"), nullable=False)
    amount = Column(Numeric(10, 2), nullable=False)  # Enforcing numeric precision over standard floats
    unit = Column(String(50), nullable=False)

    # Bidirectional relational anchors
    recipe = relationship("Recipe", back_populates="recipe_ingredients")
    ingredient = relationship("Ingredient", back_populates="recipe_ingredients")

    def __repr__(self):
        return f"<RecipeIngredient(recipe_id={self.recipe_id}, ingredient_id={self.ingredient_id}, amount={self.amount} {self.unit})>"

```

#### MealPlan Model Definition (`src/recipebox/models/meal_plan.py`)

```python
from sqlalchemy import Column, String, Integer, Date, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from src.recipebox.database import Base


class MealPlan(Base):
    __tablename__ = "meal_plans"

    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(String(255), nullable=False)
    week_start_date = Column(Date, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Placeholder relationship hook to preserve Phase 1 Task breakdowns
    # (MealPlanItem model will back-populate here in Task 2)
    meal_plan_items = relationship(
        "MealPlanItem",
        back_populates="meal_plan",
        cascade="all, delete-orphan",
        primaryjoin="MealPlan.id == foreign(MealPlanItem.meal_plan_id)"
    )

    def __repr__(self):
        return f"<MealPlan(id={self.id}, user_id='{self.user_id}', week_start={self.week_start_date})>"

```

---

### 📋 2. Automated Foundation Unit Test Suite

#### Ingredient Testing Specifications (`tests/unit/test_ingredient_model.py`)

```python
import pytest
from sqlalchemy.exc import IntegrityError
from src.recipebox.models.ingredient import Ingredient


def test_ingredient_fields(db_session):
    """Test Ingredient model has correct fields and configuration."""
    ingredient = Ingredient(
        name="Organic All-Purpose Flour",
        category="Baking",
        description="Unbleached white flour processed from organic hard wheat."
    )
    db_session.add(ingredient)
    db_session.commit()

    assert ingredient.id is not None
    assert ingredient.name == "Organic All-Purpose Flour"
    assert ingredient.category == "Baking"
    assert ingredient.description == "Unbleached white flour processed from organic hard wheat."


def test_ingredient_unique_name(db_session):
    """Test Ingredient name constraint configuration enforces unique values."""
    ing1 = Ingredient(name="Whole Milk", category="Dairy")
    ing2 = Ingredient(name="Whole Milk", category="Beverages")
    
    db_session.add(ing1)
    db_session.commit()
    
    db_session.add(ing2)
    with pytest.raises(IntegrityError):
        db_session.commit()
    db_session.rollback()


def test_ingredient_str_repr():
    """Test Ingredient string representation structure matches standards."""
    ingredient = Ingredient(name="Unsalted Butter", category="Dairy")
    string_representation = repr(ingredient)
    
    assert "Unsalted Butter" in string_representation
    assert "Dairy" in string_representation

```

#### Recipe Testing Specifications (`tests/unit/test_recipe_model.py`)

```python
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.recipe_ingredient import RecipeIngredient
from src.recipebox.models.ingredient import Ingredient


def test_recipe_fields(db_session):
    """Test Recipe model accurately persists standard parameters."""
    recipe = Recipe(
        name="Classic Garlic Shrimp",
        description="Quick pan-seared shrimp bound in emulsified garlic butter sauce.",
        prep_time=15,
        cook_time=10,
        servings=4,
        created_by="chef_teguh_37"
    )
    db_session.add(recipe)
    db_session.commit()

    assert recipe.id is not None
    assert recipe.name == "Classic Garlic Shrimp"
    assert recipe.description == "Quick pan-seared shrimp bound in emulsified garlic butter sauce."
    assert recipe.prep_time == 15
    assert recipe.cook_time == 10
    assert recipe.servings == 4
    assert recipe.created_by == "chef_teguh_37"


def test_recipe_ingredients_relationship(db_session):
    """Test Recipe safely manages its relationship lines down to RecipeIngredients."""
    recipe = Recipe(name="Simple Pasta", servings=2, created_by="user_id_101")
    ingredient = Ingredient(name="Spaghetti", category="Pantry")
    db_session.add(recipe)
    db_session.add(ingredient)
    db_session.commit()

    recipe_ingredient = RecipeIngredient(
        recipe_id=recipe.id,
        ingredient_id=ingredient.id,
        amount=200.00,
        unit="gram"
    )
    db_session.add(recipe_ingredient)
    db_session.commit()

    assert len(recipe.recipe_ingredients) == 1
    assert recipe.recipe_ingredients[0].amount == 200.00
    assert recipe.recipe_ingredients[0].unit == "gram"


def test_recipe_cascade_delete(db_session):
    """Test deleting a Recipe cascadingly purges all dependent join lines."""
    recipe = Recipe(name="Test Bake", servings=1, created_by="admin")
    ingredient = Ingredient(name="Sugar", category="Baking")
    db_session.add(recipe)
    db_session.add(ingredient)
    db_session.commit()

    ri = RecipeIngredient(recipe_id=recipe.id, ingredient_id=ingredient.id, amount=50.00, unit="gram")
    db_session.add(ri)
    db_session.commit()
    ri_id = ri.id

    db_session.delete(recipe)
    db_session.commit()

    queried_ri = db_session.query(RecipeIngredient).filter(RecipeIngredient.id == ri_id).first()
    assert queried_ri is None


def test_recipe_str_repr():
    """Test Recipe string validation maps name and serving parameters cleanly."""
    recipe = Recipe(name="Margherita Pizza", servings=3)
    string_representation = repr(recipe)
    
    assert "Margherita Pizza" in string_representation
    assert "3" in string_representation

```

#### RecipeIngredient Testing Specifications (`tests/unit/test_recipe_ingredient_model.py`)

```python
from decimal import Decimal
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.ingredient import Ingredient
from src.recipebox.models.recipe_ingredient import RecipeIngredient


def test_recipe_ingredient_fields(db_session):
    """Test RecipeIngredient maintains numeric precision constraints."""
    recipe = Recipe(name="Oatmeal", servings=1, created_by="tester")
    ingredient = Ingredient(name="Rolled Oats", category="Pantry")
    db_session.add(recipe)
    db_session.add(ingredient)
    db_session.commit()

    ri = RecipeIngredient(
        recipe_id=recipe.id,
        ingredient_id=ingredient.id,
        amount=Decimal("1.50"),
        unit="cup"
    )
    db_session.add(ri)
    db_session.commit()

    assert ri.id is not None
    assert ri.recipe_id == recipe.id
    assert ri.ingredient_id == ingredient.id
    assert ri.amount == Decimal("1.50")
    assert ri.unit == "cup"


def test_recipe_ingredient_relationships(db_session):
    """Test RecipeIngredient relationships traverse cleanly in both directions."""
    recipe = Recipe(name="Hot Chocolate", servings=1, created_by="tester")
    ingredient = Ingredient(name="Cocoa Powder", category="Pantry")
    db_session.add(recipe)
    db_session.add(ingredient)
    db_session.commit()

    ri = RecipeIngredient(recipe_id=recipe.id, ingredient_id=ingredient.id, amount=2.00, unit="tablespoon")
    db_session.add(ri)
    db_session.commit()

    assert recipe.recipe_ingredients[0].ingredient.name == "Cocoa Powder"
    assert ingredient.recipe_ingredients[0].recipe.name == "Hot Chocolate"


def test_recipe_ingredient_str_repr():
    """Test RecipeIngredient string output captures volume definitions correctly."""
    ri = RecipeIngredient(recipe_id=10, ingredient_id=20, amount=Decimal("3.25"), unit="ounce")
    string_representation = repr(ri)
    
    assert "3.25" in string_representation
    assert "ounce" in string_representation

```

#### MealPlan Testing Specifications (`tests/unit/test_meal_plan_model.py`)

```python
from datetime import date
from src.recipebox.models.meal_plan import MealPlan


def test_meal_plan_fields(db_session):
    """Test MealPlan correctly persists user identities and baseline parameters."""
    target_date = date(2026, 7, 20)
    meal_plan = MealPlan(
        user_id="user_bali_37",
        week_start_date=target_date
    )
    db_session.add(meal_plan)
    db_session.commit()

    assert meal_plan.id is not None
    assert meal_plan.user_id == "user_bali_37"
    assert meal_plan.week_start_date == target_date


def test_meal_plan_str_repr():
    """Test MealPlan format correctly stringifies dates and identity blocks."""
    target_date = date(2026, 7, 20)
    meal_plan = MealPlan(user_id="user_buleleng", week_start_date=target_date)
    string_representation = repr(meal_plan)
    
    assert "user_buleleng" in string_representation
    assert "2026-07-20" in string_representation

```

## Complete Data Layer and One Route or Orchestration Cycle

In the previous exercise you implemented four foundation models (Recipe, Ingredient, RecipeIngredient, MealPlan). Now you will complete the data layer and touch the API or orchestration workflow.

Step 1 — Remaining models: Implement MealPlanItem, ShoppingList, and ShoppingListItem with the necessary columns and relationships. Use precise decimal types (e.g. SQLAlchemy Numeric(10, 2)) for amount and total_amount; avoid Float. Add proper cascade deletes (e.g. deleting a meal plan removes its items; deleting a shopping list removes its items). Write unit tests for these models.

Step 2 — Migration: Add an Alembic migration that creates or updates all seven tables (the four from Task 1 plus these three). The project uses Alembic for versioned schema changes (see CLAUDE.md). Generate with alembic revision --autogenerate and apply with alembic upgrade head.

Step 3 — One route or one orchestration cycle: Do either (a) or (b):

    (a) One FastAPI route: Implement one endpoint (e.g. GET /api/recipes or POST /api/recipes) with a Pydantic schema and a call into a repository or service. Add a minimal integration test (e.g. TestClient) that asserts status and response shape.
    (b) One orchestration cycle: Run one task from specs/recipebox/tasks.md via the task-executor agent (e.g. T009 Recipe API endpoints), validate (run tests, spot-check code), commit with a clear message, and document in a short log (e.g. orchestration-log.md): task delegated, validation done, commit message.

Steps: 1. Implement MealPlanItem, ShoppingList, ShoppingListItem (Numeric(10,2) for amounts). 2. Add unit tests for the new models. 3. Create and apply Alembic migration for all 7 models. 4. Either implement one FastAPI route (schema + test) or run one task via task-executor and document in a short log.

Deliverables: MealPlanItem, ShoppingList, and ShoppingListItem model files and tests; one Alembic migration; and either one route (with schema and test) or one orchestration log. Once finished, you're ready for Task 3 (advanced services) and optionally Task 4 (orchestration and quality pipeline).

```
# TODO List - Complete File Contents

*Last updated: 2026-07-15*
*Total files with TODOs: 15*
*Total TODO items: 93*

---

## Models

### # src/recipebox/models/recipe.py

```python
from datetime import datetime
from sqlalchemy import Column, String, Integer, Text, DateTime, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class Recipe(Base):
    __tablename__ = "recipes"

    # TODO: Add columns for id, name, description, prep_time, cook_time, servings, created_by (FK to User)
    # TODO: Add created_at and updated_at timestamp columns

    # TODO: Add relationship to User (back_populates="recipes")
    # TODO: Add relationship to RecipeIngredient (back_populates="recipe", cascade="all, delete-orphan")
    # TODO: Add relationship to MealPlanItem (back_populates="recipe")

    def __repr__(self):
        return f"<Recipe(id={self.id}, name='{self.name}', servings={self.servings})>"
```

**TODOs:**
- Line 10: Add columns for id, name, description, prep_time, cook_time, servings, created_by (FK to User)
- Line 11: Add created_at and updated_at timestamp columns
- Line 13: Add relationship to User (back_populates="recipes")
- Line 14: Add relationship to RecipeIngredient (back_populates="recipe", cascade="all, delete-orphan")
- Line 15: Add relationship to MealPlanItem (back_populates="recipe")

---

### # src/recipebox/models/meal_plan.py

```python
from datetime import datetime
from sqlalchemy import Column, String, Integer, Date, DateTime, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class MealPlan(Base):
    __tablename__ = "meal_plans"

    # TODO: Add columns for id, user_id (FK), name, week_start_date
    # TODO: Add created_at and updated_at timestamp columns

    # TODO: Add relationship to User (back_populates="meal_plans")
    # TODO: Add relationship to MealPlanItem (back_populates="meal_plan", cascade="all, delete-orphan")
    # TODO: Add relationship to ShoppingList (back_populates="meal_plan", cascade="all, delete-orphan")

    def __repr__(self):
        return f"<MealPlan(id={self.id}, name='{self.name}', week_start={self.week_start_date})>"
```

**TODOs:**
- Line 10: Add columns for id, user_id (FK), name, week_start_date
- Line 11: Add created_at and updated_at timestamp columns
- Line 13: Add relationship to User (back_populates="meal_plans")
- Line 14: Add relationship to MealPlanItem (back_populates="meal_plan", cascade="all, delete-orphan")
- Line 15: Add relationship to ShoppingList (back_populates="meal_plan", cascade="all, delete-orphan")

---

### # src/recipebox/models/recipe_ingredient.py

```python
from sqlalchemy import Column, Integer, String, Numeric, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class RecipeIngredient(Base):
    __tablename__ = "recipe_ingredients"

    # TODO: Add columns for id, recipe_id (FK), ingredient_id (FK), amount, unit

    # TODO: Add relationship to Recipe (back_populates="recipe_ingredients")
    # TODO: Add relationship to Ingredient (back_populates="recipe_ingredients")

    def __repr__(self):
        return f"<RecipeIngredient(recipe_id={self.recipe_id}, ingredient_id={self.ingredient_id}, amount={self.amount} {self.unit})>"
```

**TODOs:**
- Line 9: Add columns for id, recipe_id (FK), ingredient_id (FK), amount, unit
- Line 11: Add relationship to Recipe (back_populates="recipe_ingredients")
- Line 12: Add relationship to Ingredient (back_populates="recipe_ingredients")

---

### # src/recipebox/models/ingredient.py

```python
from sqlalchemy import Column, String, Integer, Text
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class Ingredient(Base):
    __tablename__ = "ingredients"

    # TODO: Add columns for id, name (unique), category, description

    # TODO: Add relationship to RecipeIngredient (back_populates="ingredient")
    # TODO: Add relationship to ShoppingListItem (back_populates="ingredient")

    def __repr__(self):
        return f"<Ingredient(id={self.id}, name='{self.name}', category='{self.category}')>"
```

**TODOs:**
- Line 9: Add columns for id, name (unique), category, description
- Line 11: Add relationship to RecipeIngredient (back_populates="ingredient")
- Line 12: Add relationship to ShoppingListItem (back_populates="ingredient")

---

### # src/recipebox/models/meal_plan_item.py

```python
from sqlalchemy import Column, Integer, String, Date, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class MealPlanItem(Base):
    __tablename__ = "meal_plan_items"

    # TODO: Add columns for id, meal_plan_id (FK), recipe_id (FK), date, meal_type, servings

    # TODO: Add relationship to MealPlan (back_populates="meal_plan_items")
    # TODO: Add relationship to Recipe (back_populates="meal_plan_items")

    def __repr__(self):
        return f"<MealPlanItem(id={self.id}, date={self.date}, meal_type='{self.meal_type}', servings={self.servings})>"
```

**TODOs:**
- Line 9: Add columns for id, meal_plan_id (FK), recipe_id (FK), date, meal_type, servings
- Line 11: Add relationship to MealPlan (back_populates="meal_plan_items")
- Line 12: Add relationship to Recipe (back_populates="meal_plan_items")

---

### # src/recipebox/models/shopping_list.py

```python
from datetime import datetime
from sqlalchemy import Column, String, Integer, DateTime, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class ShoppingList(Base):
    __tablename__ = "shopping_lists"

    # TODO: Add columns for id, meal_plan_id (FK), name, created_at

    # TODO: Add relationship to MealPlan (back_populates="shopping_lists")
    # TODO: Add relationship to ShoppingListItem (back_populates="shopping_list", cascade="all, delete-orphan")

    def __repr__(self):
        return f"<ShoppingList(id={self.id}, name='{self.name}')>"
```

**TODOs:**
- Line 10: Add columns for id, meal_plan_id (FK), name, created_at
- Line 12: Add relationship to MealPlan (back_populates="shopping_lists")
- Line 13: Add relationship to ShoppingListItem (back_populates="shopping_list", cascade="all, delete-orphan")

---

### # src/recipebox/models/shopping_list_item.py

```python
from sqlalchemy import Column, Integer, String, Numeric, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class ShoppingListItem(Base):
    __tablename__ = "shopping_list_items"

    # TODO: Add columns for id, shopping_list_id (FK), ingredient_id (FK), amount, unit, category

    # TODO: Add relationship to ShoppingList (back_populates="shopping_list_items")
    # TODO: Add relationship to Ingredient (back_populates="shopping_list_items")

    def __repr__(self):
        return f"<ShoppingListItem(ingredient_id={self.ingredient_id}, amount={self.amount} {self.unit})>"
```

**TODOs:**
- Line 9: Add columns for id, shopping_list_id (FK), ingredient_id (FK), amount, unit, category
- Line 11: Add relationship to ShoppingList (back_populates="shopping_list_items")
- Line 12: Add relationship to Ingredient (back_populates="shopping_list_items")

---

## Tests

### # tests/unit/test_ingredient_model.py

```python
import pytest
from src.recipebox.models.ingredient import Ingredient


def test_ingredient_fields():
    """Test Ingredient model has all required fields"""
    # TODO: Create an Ingredient instance with name, category, and description
    # TODO: Assert each field has the correct value
    pass


def test_ingredient_recipe_ingredients_relationship():
    """Test Ingredient has relationship to RecipeIngredients"""
    # TODO: Create an Ingredient instance
    # TODO: Assert it has a 'recipe_ingredients' attribute that is an empty list
    pass


def test_ingredient_shopping_list_items_relationship():
    """Test Ingredient has relationship to ShoppingListItems"""
    # TODO: Create an Ingredient instance
    # TODO: Assert it has a 'shopping_list_items' attribute that is an empty list
    pass


def test_ingredient_str_repr():
    """Test Ingredient string representation"""
    # TODO: Create an Ingredient instance with id, name, and category
    # TODO: Assert the repr contains the name and category
    pass
```

**TODOs:**
- Line 7: Create an Ingredient instance with name, category, and description
- Line 8: Assert each field has the correct value
- Line 14: Create an Ingredient instance
- Line 15: Assert it has a 'recipe_ingredients' attribute that is an empty list
- Line 21: Create an Ingredient instance
- Line 22: Assert it has a 'shopping_list_items' attribute that is an empty list
- Line 28: Create an Ingredient instance with id, name, and category
- Line 29: Assert the repr contains the name and category

---

### # tests/unit/test_recipe_model.py

```python
import pytest
from src.recipebox.models.recipe import Recipe
from src.recipebox.models.recipe_ingredient import RecipeIngredient


def test_recipe_fields():
    """Test Recipe model has all required fields"""
    # TODO: Create a Recipe instance with all fields (name, description, prep_time, cook_time, servings, created_by)
    # TODO: Assert each field has the correct value
    pass


def test_recipe_user_relationship():
    """Test Recipe has relationship to User"""
    # TODO: Create a Recipe instance
    # TODO: Assert it has a 'user' attribute
    pass


def test_recipe_ingredients_relationship():
    """Test Recipe has relationship to RecipeIngredients"""
    # TODO: Create a Recipe instance
    # TODO: Assert it has a 'recipe_ingredients' attribute that is an empty list
    pass


def test_recipe_timestamps():
    """Test Recipe has created_at and updated_at"""
    # TODO: Create a Recipe instance
    # TODO: Assert it has 'created_at' and 'updated_at' attributes
    pass


def test_recipe_str_repr():
    """Test Recipe string representation"""
    # TODO: Create a Recipe instance with id, name, and servings
    # TODO: Assert the repr contains the name and servings
    pass
```

**TODOs:**
- Line 8: Create a Recipe instance with all fields (name, description, prep_time, cook_time, servings, created_by)
- Line 9: Assert each field has the correct value
- Line 15: Create a Recipe instance
- Line 16: Assert it has a 'user' attribute
- Line 22: Create a Recipe instance
- Line 23: Assert it has a 'recipe_ingredients' attribute that is an empty list
- Line 29: Create a Recipe instance
- Line 30: Assert it has 'created_at' and 'updated_at' attributes
- Line 36: Create a Recipe instance with id, name, and servings
- Line 37: Assert the repr contains the name and servings

---

### # tests/unit/test_recipe_ingredient_model.py

```python
import pytest
from src.recipebox.models.recipe_ingredient import RecipeIngredient


def test_recipe_ingredient_fields():
    """Test RecipeIngredient model has all required fields"""
    # TODO: Create a RecipeIngredient instance with recipe_id, ingredient_id, amount, and unit
    # TODO: Assert each field has the correct value
    pass


def test_recipe_ingredient_recipe_relationship():
    """Test RecipeIngredient has relationship to Recipe"""
    # TODO: Create a RecipeIngredient instance
    # TODO: Assert it has a 'recipe' attribute
    pass


def test_recipe_ingredient_ingredient_relationship():
    """Test RecipeIngredient has relationship to Ingredient"""
    # TODO: Create a RecipeIngredient instance
    # TODO: Assert it has an 'ingredient' attribute
    pass


def test_recipe_ingredient_str_repr():
    """Test RecipeIngredient string representation"""
    # TODO: Create a RecipeIngredient instance
    # TODO: Assert the repr contains the amount and unit
    pass
```

**TODOs:**
- Line 7: Create a RecipeIngredient instance with recipe_id, ingredient_id, amount, and unit
- Line 8: Assert each field has the correct value
- Line 14: Create a RecipeIngredient instance
- Line 15: Assert it has a 'recipe' attribute
- Line 21: Create a RecipeIngredient instance
- Line 22: Assert it has an 'ingredient' attribute
- Line 28: Create a RecipeIngredient instance
- Line 29: Assert the repr contains the amount and unit

---

### # tests/unit/test_meal_plan_model.py

```python
import pytest
from datetime import date
from src.recipebox.models.meal_plan import MealPlan


def test_meal_plan_fields():
    """Test MealPlan model has all required fields"""
    # TODO: Create a MealPlan instance with user_id, name, and week_start_date
    # TODO: Assert each field has the correct value
    pass


def test_meal_plan_user_relationship():
    """Test MealPlan has relationship to User"""
    # TODO: Create a MealPlan instance
    # TODO: Assert it has a 'user' attribute
    pass


def test_meal_plan_items_relationship():
    """Test MealPlan has relationship to MealPlanItems"""
    # TODO: Create a MealPlan instance
    # TODO: Assert it has a 'meal_plan_items' attribute that is an empty list
    pass


def test_meal_plan_shopping_lists_relationship():
    """Test MealPlan has relationship to ShoppingLists"""
    # TODO: Create a MealPlan instance
    # TODO: Assert it has a 'shopping_lists' attribute that is an empty list
    pass


def test_meal_plan_timestamps():
    """Test MealPlan has created_at and updated_at"""
    # TODO: Create a MealPlan instance
    # TODO: Assert it has 'created_at' and 'updated_at' attributes
    pass


def test_meal_plan_str_repr():
    """Test MealPlan string representation"""
    # TODO: Create a MealPlan instance with id, name, and week_start_date
    # TODO: Assert the repr contains the name
    pass
```

**TODOs:**
- Line 8: Create a MealPlan instance with user_id, name, and week_start_date
- Line 9: Assert each field has the correct value
- Line 15: Create a MealPlan instance
- Line 16: Assert it has a 'user' attribute
- Line 22: Create a MealPlan instance
- Line 23: Assert it has a 'meal_plan_items' attribute that is an empty list
- Line 29: Create a MealPlan instance
- Line 30: Assert it has a 'shopping_lists' attribute that is an empty list
- Line 36: Create a MealPlan instance
- Line 37: Assert it has 'created_at' and 'updated_at' attributes
- Line 43: Create a MealPlan instance with id, name, and week_start_date
- Line 44: Assert the repr contains the name

---

### # tests/unit/test_meal_plan_item_model.py

```python
import pytest
from datetime import date
from src.recipebox.models.meal_plan_item import MealPlanItem


def test_meal_plan_item_fields():
    """Test MealPlanItem model has all required fields"""
    # TODO: Create a MealPlanItem instance with meal_plan_id, recipe_id, date, meal_type, and servings
    # TODO: Assert each field has the correct value
    pass


def test_meal_plan_item_meal_plan_relationship():
    """Test MealPlanItem has relationship to MealPlan"""
    # TODO: Create a MealPlanItem instance
    # TODO: Assert it has a 'meal_plan' attribute
    pass


def test_meal_plan_item_recipe_relationship():
    """Test MealPlanItem has relationship to Recipe"""
    # TODO: Create a MealPlanItem instance
    # TODO: Assert it has a 'recipe' attribute
    pass


def test_meal_plan_item_str_repr():
    """Test MealPlanItem string representation"""
    # TODO: Create a MealPlanItem instance
    # TODO: Assert the repr contains the meal_type and servings
    pass
```

**TODOs:**
- Line 8: Create a MealPlanItem instance with meal_plan_id, recipe_id, date, meal_type, and servings
- Line 9: Assert each field has the correct value
- Line 15: Create a MealPlanItem instance
- Line 16: Assert it has a 'meal_plan' attribute
- Line 22: Create a MealPlanItem instance
- Line 23: Assert it has a 'recipe' attribute
- Line 29: Create a MealPlanItem instance
- Line 30: Assert the repr contains the meal_type and servings

---

### # tests/unit/test_shopping_list_model.py

```python
import pytest
from src.recipebox.models.shopping_list import ShoppingList


def test_shopping_list_fields():
    """Test ShoppingList model has all required fields"""
    # TODO: Create a ShoppingList instance with meal_plan_id and name
    # TODO: Assert each field has the correct value
    pass


def test_shopping_list_meal_plan_relationship():
    """Test ShoppingList has relationship to MealPlan"""
    # TODO: Create a ShoppingList instance
    # TODO: Assert it has a 'meal_plan' attribute
    pass


def test_shopping_list_items_relationship():
    """Test ShoppingList has relationship to ShoppingListItems"""
    # TODO: Create a ShoppingList instance
    # TODO: Assert it has a 'shopping_list_items' attribute that is an empty list
    pass


def test_shopping_list_timestamps():
    """Test ShoppingList has created_at"""
    # TODO: Create a ShoppingList instance
    # TODO: Assert it has a 'created_at' attribute
    pass


def test_shopping_list_str_repr():
    """Test ShoppingList string representation"""
    # TODO: Create a ShoppingList instance with id and name
    # TODO: Assert the repr contains the name
    pass
```

**TODOs:**
- Line 7: Create a ShoppingList instance with meal_plan_id and name
- Line 8: Assert each field has the correct value
- Line 14: Create a ShoppingList instance
- Line 15: Assert it has a 'meal_plan' attribute
- Line 21: Create a ShoppingList instance
- Line 22: Assert it has a 'shopping_list_items' attribute that is an empty list
- Line 28: Create a ShoppingList instance
- Line 29: Assert it has a 'created_at' attribute
- Line 35: Create a ShoppingList instance with id and name
- Line 36: Assert the repr contains the name

---

### # tests/unit/test_shopping_list_item_model.py

```python
import pytest
from src.recipebox.models.shopping_list_item import ShoppingListItem


def test_shopping_list_item_fields():
    """Test ShoppingListItem model has all required fields"""
    # TODO: Create a ShoppingListItem instance with shopping_list_id, ingredient_id, amount, unit, and category
    # TODO: Assert each field has the correct value
    pass


def test_shopping_list_item_shopping_list_relationship():
    """Test ShoppingListItem has relationship to ShoppingList"""
    # TODO: Create a ShoppingListItem instance
    # TODO: Assert it has a 'shopping_list' attribute
    pass


def test_shopping_list_item_ingredient_relationship():
    """Test ShoppingListItem has relationship to Ingredient"""
    # TODO: Create a ShoppingListItem instance
    # TODO: Assert it has an 'ingredient' attribute
    pass


def test_shopping_list_item_str_repr():
    """Test ShoppingListItem string representation"""
    # TODO: Create a ShoppingListItem instance
    # TODO: Assert the repr contains the amount and unit
    pass
```

**TODOs:**
- Line 7: Create a ShoppingListItem instance with shopping_list_id, ingredient_id, amount, unit, and category
- Line 8: Assert each field has the correct value
- Line 14: Create a ShoppingListItem instance
- Line 15: Assert it has a 'shopping_list' attribute
- Line 21: Create a ShoppingListItem instance
- Line 22: Assert it has an 'ingredient' attribute
- Line 28: Create a ShoppingListItem instance
- Line 29: Assert the repr contains the amount and unit

---

## Database Migrations

### # alembic/versions/001_initial_models.py

```python
"""Initial models

Revision ID: 001
Revises:
Create Date: 2024-12-01 10:00:00.000000

"""
from alembic import op
import sqlalchemy as sa


# revision identifiers, used by Alembic.
revision = '001'
down_revision = None
branch_labels = None
depends_on = None


def upgrade():
    # TODO: Create recipes table with all columns (id, name, description, prep_time, cook_time, servings, created_by FK, timestamps)
    # TODO: Create ingredients table with all columns (id, name unique, category, description)
    # TODO: Create recipe_ingredients table with all columns (id, recipe_id FK, ingredient_id FK, amount, unit)
    # TODO: Create meal_plans table with all columns (id, user_id FK, name, week_start_date, timestamps)
    # TODO: Create meal_plan_items table with all columns (id, meal_plan_id FK, recipe_id FK, date, meal_type, servings)
    # TODO: Create shopping_lists table with all columns (id, meal_plan_id FK, name, created_at)
    # TODO: Create shopping_list_items table with all columns (id, shopping_list_id FK, ingredient_id FK, amount, unit, category)
    pass


def downgrade():
    # TODO: Drop all tables in reverse order (shopping_list_items, shopping_lists, meal_plan_items, meal_plans, recipe_ingredients, ingredients, recipes)
    pass
```

**TODOs:**
- Line 20: Create recipes table with all columns (id, name, description, prep_time, cook_time, servings, created_by FK, timestamps)
- Line 21: Create ingredients table with all columns (id, name unique, category, description)
- Line 22: Create recipe_ingredients table with all columns (id, recipe_id FK, ingredient_id FK, amount, unit)
- Line 23: Create meal_plans table with all columns (id, user_id FK, name, week_start_date, timestamps)
- Line 24: Create meal_plan_items table with all columns (id, meal_plan_id FK, recipe_id FK, date, meal_type, servings)
- Line 25: Create shopping_lists table with all columns (id, meal_plan_id FK, name, created_at)
- Line 26: Create shopping_list_items table with all columns (id, shopping_list_id FK, ingredient_id FK, amount, unit, category)
- Line 31: Drop all tables in reverse order (shopping_list_items, shopping_lists, meal_plan_items, meal_plans, recipe_ingredients, ingredients, recipes)

---

## Summary

This RecipeBox project has 93 TODO items across 15 files that need to be completed:

**Models (7 files, 23 TODOs):**
- Database column definitions needed
- SQLAlchemy relationships need to be defined

**Tests (7 files, 62 TODOs):**
- Unit tests need to be implemented
- Test coverage for all model fields and relationships

**Migrations (1 file, 8 TODOs):**
- Alembic migration scripts for table creation and deletion


```

Here is the complete, production-ready, placeholder-free implementation of the remaining three core data layer models (`MealPlanItem`, `ShoppingList`, `ShoppingListItem`), a unified `001_initial_models.py` Alembic migration script handling all seven entities, and choice **(a)**—the FastAPI route setup including Pydantic input/output schemas alongside a complete test verification layer using `FastAPI TestClient`.

---

## 🛠️ Phase 1 & 2: Complete Data Layer Models

### 1. MealPlanItem Model (`src/recipebox/models/meal_plan_item.py`)

```python
from sqlalchemy import Column, Integer, String, Date, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class MealPlanItem(Base):
    __tablename__ = "meal_plan_items"

    id = Column(Integer, primary_key=True, index=True)
    meal_plan_id = Column(Integer, ForeignKey("meal_plans.id", ondelete="CASCADE"), nullable=False)
    recipe_id = Column(Integer, ForeignKey("recipes.id", ondelete="CASCADE"), nullable=False)
    date = Column(Date, nullable=False)
    meal_type = Column(String(50), nullable=False)  # e.g., Breakfast, Lunch, Dinner, Snack
    servings = Column(Integer, nullable=False)

    # Relationships mapped back cleanly to parent models
    meal_plan = relationship("MealPlan", back_populates="meal_plan_items")
    recipe = relationship("Recipe", back_populates="meal_plan_items")

    def __repr__(self):
        return f"<MealPlanItem(id={self.id}, date={self.date}, meal_type='{self.meal_type}', servings={self.servings})>"

```

### 2. ShoppingList Model (`src/recipebox/models/shopping_list.py`)

```python
from datetime import datetime
from sqlalchemy import Column, String, Integer, DateTime, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class ShoppingList(Base):
    __tablename__ = "shopping_lists"

    id = Column(Integer, primary_key=True, index=True)
    meal_plan_id = Column(Integer, ForeignKey("meal_plans.id", ondelete="CASCADE"), nullable=False)
    name = Column(String(255), nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow, nullable=False)

    # Relationships
    meal_plan = relationship("MealPlan", back_populates="shopping_lists")
    shopping_list_items = relationship(
        "ShoppingListItem", 
        back_populates="shopping_list", 
        cascade="all, delete-orphan"
    )

    def __repr__(self):
        return f"<ShoppingList(id={self.id}, name='{self.name}')>"

```

### 3. ShoppingListItem Model (`src/recipebox/models/shopping_list_item.py`)

```python
from sqlalchemy import Column, Integer, String, Numeric, ForeignKey
from sqlalchemy.orm import relationship
from src.recipebox.database import Base


class ShoppingListItem(Base):
    __tablename__ = "shopping_list_items"

    id = Column(Integer, primary_key=True, index=True)
    shopping_list_id = Column(Integer, ForeignKey("shopping_lists.id", ondelete="CASCADE"), nullable=False)
    ingredient_id = Column(Integer, ForeignKey("ingredients.id", ondelete="CASCADE"), nullable=False)
    amount = Column(Numeric(10, 2), nullable=False)  # Precise decimals to avoid float rounding errors
    unit = Column(String(50), nullable=False)
    category = Column(String(100), nullable=True)

    # Bidirectional relationships
    shopping_list = relationship("ShoppingList", back_populates="shopping_list_items")
    ingredient = relationship("Ingredient", back_populates="shopping_list_items")

    def __repr__(self):
        return f"<ShoppingListItem(ingredient_id={self.ingredient_id}, amount={self.amount} {self.unit})>"

```

---

## 🧪 Phase 3: Structural Model Unit Tests

### 1. MealPlanItem Tests (`tests/unit/test_meal_plan_item_model.py`)

```python
import pytest
from datetime import date
from src.recipebox.models.meal_plan_item import MealPlanItem


def test_meal_plan_item_fields():
    """Test MealPlanItem model has all required fields."""
    item = MealPlanItem(meal_plan_id=1, recipe_id=2, date=date(2026, 7, 15), meal_type="Dinner", servings=4)
    assert item.meal_plan_id == 1
    assert item.recipe_id == 2
    assert item.date == date(2026, 7, 15)
    assert item.meal_type == "Dinner"
    assert item.servings == 4


def test_meal_plan_item_meal_plan_relationship():
    """Test MealPlanItem has relationship to MealPlan."""
    item = MealPlanItem()
    assert hasattr(item, "meal_plan")


def test_meal_plan_item_recipe_relationship():
    """Test MealPlanItem has relationship to Recipe."""
    item = MealPlanItem()
    assert hasattr(item, "recipe")


def test_meal_plan_item_str_repr():
    """Test MealPlanItem string representation."""
    item = MealPlanItem(id=5, date=date(2026, 7, 15), meal_type="Lunch", servings=2)
    assert "Lunch" in repr(item) and "2" in repr(item)

```

### 2. ShoppingList Tests (`tests/unit/test_shopping_list_model.py`)

```python
import pytest
from src.recipebox.models.shopping_list import ShoppingList


def test_shopping_list_fields():
    """Test ShoppingList model has all required fields."""
    sl = ShoppingList(meal_plan_id=1, name="Weekly List")
    assert sl.meal_plan_id == 1
    assert sl.name == "Weekly List"


def test_shopping_list_meal_plan_relationship():
    """Test ShoppingList has relationship to MealPlan."""
    sl = ShoppingList()
    assert hasattr(sl, "meal_plan")


def test_shopping_list_items_relationship():
    """Test ShoppingList has relationship to ShoppingListItems."""
    sl = ShoppingList()
    assert hasattr(sl, "shopping_list_items")
    assert isinstance(sl.shopping_list_items, list)


def test_shopping_list_timestamps():
    """Test ShoppingList has created_at."""
    sl = ShoppingList()
    assert hasattr(sl, "created_at")


def test_shopping_list_str_repr():
    """Test ShoppingList string representation."""
    sl = ShoppingList(id=1, name="Grocery Run")
    assert "Grocery Run" in repr(sl)

```

### 3. ShoppingListItem Tests (`tests/unit/test_shopping_list_item_model.py`)

```python
import pytest
from decimal import Decimal
from src.recipebox.models.shopping_list_item import ShoppingListItem


def test_shopping_list_item_fields():
    """Test ShoppingListItem model has all required fields."""
    sli = ShoppingListItem(shopping_list_id=1, ingredient_id=2, amount=Decimal("2.50"), unit="cups", category="Produce")
    assert sli.shopping_list_id == 1
    assert sli.ingredient_id == 2
    assert sli.amount == Decimal("2.50")
    assert sli.unit == "cups"
    assert sli.category == "Produce"


def test_shopping_list_item_shopping_list_relationship():
    """Test ShoppingListItem has relationship to ShoppingList."""
    sli = ShoppingListItem()
    assert hasattr(sli, "shopping_list")


def test_shopping_list_item_ingredient_relationship():
    """Test ShoppingListItem has relationship to Ingredient."""
    sli = ShoppingListItem()
    assert hasattr(sli, "ingredient")


def test_shopping_list_item_str_repr():
    """Test ShoppingListItem string representation."""
    sli = ShoppingListItem(ingredient_id=3, amount=Decimal("150.00"), unit="grams")
    assert "150.00" in repr(sli) and "grams" in repr(sli)

```

---

## 🗺️ Phase 4: Definitive Alembic Schema Database Migration

### Initial Configuration Script (`alembic/versions/001_initial_models.py`)

```python
"""Initial models

Revision ID: 001
Revises:
Create Date: 2026-07-15 10:00:00.000000

"""
from alembic import op
import sqlalchemy as sa


# revision identifiers, used by Alembic.
revision = '001'
down_revision = None
branch_labels = None
depends_on = None


def upgrade():
    # 1. Create Core Master 'ingredients' Table
    op.create_table(
        'ingredients',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('name', sa.String(length=255), nullable=False, unique=True),
        sa.Column('category', sa.String(length=100), nullable=True),
        sa.Column('description', sa.Text(), nullable=True),
        sa.PrimaryKeyConstraint('id')
    )
    
    # 2. Create Core Master 'recipes' Table
    op.create_table(
        'recipes',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('name', sa.String(length=255), nullable=False),
        sa.Column('description', sa.Text(), nullable=True),
        sa.Column('prep_time', sa.Integer(), nullable=True),
        sa.Column('cook_time', sa.Integer(), nullable=True),
        sa.Column('servings', sa.Integer(), nullable=False),
        sa.Column('created_by', sa.String(length=255), nullable=False),
        sa.Column('created_at', sa.DateTime(), nullable=False, server_default=sa.func.now()),
        sa.Column('updated_at', sa.DateTime(), nullable=False, server_default=sa.func.now(), server_onupdate=sa.func.now()),
        sa.PrimaryKeyConstraint('id')
    )

    # 3. Create 'recipe_ingredients' Many-to-Many Dependent Table
    op.create_table(
        'recipe_ingredients',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('recipe_id', sa.Integer(), nullable=False),
        sa.Column('ingredient_id', sa.Integer(), nullable=False),
        sa.Column('amount', sa.Numeric(precision=10, scale=2), nullable=False),
        sa.Column('unit', sa.String(length=50), nullable=False),
        sa.ForeignKeyConstraint(['recipe_id'], ['recipes.id'], ondelete='CASCADE'),
        sa.ForeignKeyConstraint(['ingredient_id'], ['ingredients.id'], ondelete='CASCADE'),
        sa.PrimaryKeyConstraint('id')
    )

    # 4. Create Master Calendar 'meal_plans' Table
    op.create_table(
        'meal_plans',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('user_id', sa.String(length=255), nullable=False),
        sa.Column('name', sa.String(length=255), nullable=True),
        sa.Column('week_start_date', sa.Date(), nullable=False),
        sa.Column('created_at', sa.DateTime(), nullable=False, server_default=sa.func.now()),
        sa.Column('updated_at', sa.DateTime(), nullable=False, server_default=sa.func.now(), server_onupdate=sa.func.now()),
        sa.PrimaryKeyConstraint('id')
    )

    # 5. Create 'meal_plan_items' Dependent Schedule Table
    op.create_table(
        'meal_plan_items',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('meal_plan_id', sa.Integer(), nullable=False),
        sa.Column('recipe_id', sa.Integer(), nullable=False),
        sa.Column('date', sa.Date(), nullable=False),
        sa.Column('meal_type', sa.String(length=50), nullable=False),
        sa.Column('servings', sa.Integer(), nullable=False),
        sa.ForeignKeyConstraint(['meal_plan_id'], ['meal_plans.id'], ondelete='CASCADE'),
        sa.ForeignKeyConstraint(['recipe_id'], ['recipes.id'], ondelete='CASCADE'),
        sa.PrimaryKeyConstraint('id')
    )

    # 6. Create 'shopping_lists' Master Aggregator Table
    op.create_table(
        'shopping_lists',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('meal_plan_id', sa.Integer(), nullable=False),
        sa.Column('name', sa.String(length=255), nullable=False),
        sa.Column('created_at', sa.DateTime(), nullable=False, server_default=sa.func.now()),
        sa.ForeignKeyConstraint(['meal_plan_id'], ['meal_plans.id'], ondelete='CASCADE'),
        sa.PrimaryKeyConstraint('id')
    )

    # 7. Create 'shopping_list_items' Dependent Line-Item Table
    op.create_table(
        'shopping_list_items',
        sa.Column('id', sa.Integer(), primary_key=True, autoincrement=True, nullable=False),
        sa.Column('shopping_list_id', sa.Integer(), nullable=False),
        sa.Column('ingredient_id', sa.Integer(), nullable=False),
        sa.Column('amount', sa.Numeric(precision=10, scale=2), nullable=False),
        sa.Column('unit', sa.String(length=50), nullable=False),
        sa.Column('category', sa.String(length=100), nullable=True),
        sa.ForeignKeyConstraint(['shopping_list_id'], ['shopping_lists.id'], ondelete='CASCADE'),
        sa.ForeignKeyConstraint(['ingredient_id'], ['ingredients.id'], ondelete='CASCADE'),
        sa.PrimaryKeyConstraint('id')
    )


def downgrade():
    # Drop tables in exact reverse dependency order to preserve key constraints
    op.drop_table('shopping_list_items')
    op.drop_table('shopping_lists')
    op.drop_table('meal_plan_items')
    op.drop_table('meal_plans')
    op.drop_table('recipe_ingredients')
    op.drop_table('recipes')
    op.drop_table('ingredients')

```

---

## 🚀 Options (a): FastAPI Route Delivery Layer & Integration Tests

### 1. Pydantic Schemas (`src/recipebox/schemas/recipe.py`)

```python
from pydantic import BaseModel, Field
from typing import Optional


class RecipeCreate(BaseModel):
    name: str = Field(..., min_length=1, max_length=255, example="Ayam Betutu Bali")
    description: Optional[str] = Field(None, max_length=1000)
    prep_time: Optional[int] = Field(None, ge=0, description="Preparation time in minutes")
    cook_time: Optional[int] = Field(None, ge=0, description="Cooking time in minutes")
    servings: int = Field(..., ge=1, le=50, description="Number of target servings")
    created_by: str = Field(..., min_length=1, max_length=255)


class RecipeResponse(BaseModel):
    id: int
    name: str
    description: Optional[str]
    prep_time: Optional[int]
    cook_time: Optional[int]
    servings: int
    created_by: str

    class Config:
        from_attributes = True  # Allows lazy conversion from SQLAlchemy ORM records

```

### 2. Functional API Router Handler Route (`src/recipebox/routers/recipes.py`)

```python
from fastapi import APIRouter, Depends, HTTPException, status
from sqlalchemy.orm import Session
from src.recipebox.database import get_db
from src.recipebox.models.recipe import Recipe
from src.recipebox.schemas.recipe import RecipeCreate, RecipeResponse

router = APIRouter(prefix="/api/recipes", tags=["Recipes"])


@router.post("/", response_model=RecipeResponse, status_code=status.HTTP_201_CREATED)
def create_recipe(payload: RecipeCreate, db: Session = Depends(get_db)):
    """Creates a new Recipe record inside the centralized database stack."""
    # Simulating standard Repository Pattern layer persistence
    db_recipe = Recipe(
        name=payload.name,
        description=payload.description,
        prep_time=payload.prep_time,
        cook_time=payload.cook_time,
        servings=payload.servings,
        created_by=payload.created_by
    )
    db.add(db_recipe)
    db.commit()
    db.refresh(db_recipe)
    return db_recipe

```

### 3. API Integration Test Layer (`tests/integration/test_recipes_api.py`)

```python
import pytest
from fastapi.testclient import TestClient
from src.recipebox.main import app

client = TestClient(app)


def test_create_recipe_api_success(db_session):
    """Test integration endpoint successfully validates input data and inserts recipe entries."""
    payload = {
        "name": "Nasi Goreng Kampoeng",
        "description": "Authentic Indonesian traditional fried rice.",
        "prep_time": 10,
        "cook_time": 15,
        "servings": 2,
        "created_by": "bagus_teguh"
    }
    
    response = client.post("/api/recipes/", json=payload)
    
    assert response.status_code == 201
    json_data = response.json()
    assert json_data["name"] == "Nasi Goreng Kampoeng"
    assert json_data["servings"] == 2
    assert "id" in json_data


def test_create_recipe_api_validation_failure():
    """Test validation boundaries fail when required parameter definitions are out of bounds."""
    invalid_payload = {
        "name": "",  # Triggers empty text validation error
        "servings": 0,  # Fails lower limit boundary target
        "created_by": "bagus_teguh"
    }
    
    response = client.post("/api/recipes/", json=invalid_payload)
    assert response.status_code == 422  # Standard FastAPI validation exception code

```



## Implement Advanced Recipe Services

You've already built the foundation models and (in Task 2) completed the data layer and either one route or one orchestration cycle. Now you add the most complex business logic in RecipeBox: shopping list generation with ingredient aggregation.

You will implement three critical services that bring the application together. The ShoppingListService handles the trickiest part — taking all meals from a meal plan, extracting their ingredients (scaled by servings), and intelligently aggregating them. For example, if three recipes all use flour in different units (cups, grams, ounces), your code must convert everything to a common unit and sum the amounts.

The SearchService enables users to find recipes by name or description, with optional filters for cooking times and servings. The NutritionService calculates nutritional totals for recipes and entire meal plans by summing ingredient nutrition values and scaling them appropriately.

Your work centers on completing the core logic in each service:

    ShoppingListService: Fill in the ingredient aggregation loop and the unit-based aggregation logic.
    SearchService: Build the query with search filters and proper ordering.
    NutritionService: Calculate nutrition totals from ingredients and scale them correctly.

Each service includes comprehensive tests to verify that your implementation handles edge cases like unit conversions, empty data, and scaling factors. Once complete, you will have proven that orchestrated development can handle complex business logic efficiently! In production, services should access data through repository methods (e.g. meal_plan_repo.get_by_id, meal_plan_repo.get_items) per CLAUDE.md; solution snippets that use the session directly are simplified for brevity.

Steps: 1. Implement ShoppingListService (aggregation loop, unit conversion, 6-step algorithm). 2. Implement SearchService (query, filters, ordering). 3. Implement NutritionService (totals, scaling). 4. Add or extend unit tests for each service (edge cases). 5. Run full test suite and fix any failures.

Deliverables: ShoppingListService, SearchService, NutritionService (and tests). Optionally run test-enhancer on one service and document changes.

```
# shopping_list_service.py
from typing import List, Dict
from decimal import Decimal
from sqlalchemy.orm import Session
from src.recipebox.models import ShoppingList, ShoppingListItem, MealPlan, MealPlanItem, RecipeIngredient, Ingredient
from src.recipebox.repositories.shopping_list_repository import ShoppingListRepository


class UnitConverter:
    """Handles unit conversions for ingredient aggregation."""
    
    # Conversion factors to base metric units (grams or milliliters)
    CONVERSIONS = {
        # Weight conversions to grams
        'kg': 1000.0,
        'g': 1.0,
        'lb': 453.592,
        'oz': 28.3495,
        
        # Volume conversions to milliliters
        'l': 1000.0,
        'ml': 1.0,
        'cup': 236.588,
        'tbsp': 14.7868,
        'tsp': 4.92892,
        
        # Piece-based (no conversion)
        'piece': 1.0,
        'unit': 1.0,
    }
    
    WEIGHT_UNITS = {'kg', 'g', 'lb', 'oz'}
    VOLUME_UNITS = {'l', 'ml', 'cup', 'tbsp', 'tsp'}
    PIECE_UNITS = {'piece', 'unit'}
    
    @classmethod
    def convert_to_base_unit(cls, amount: float, unit: str) -> tuple[float, str]:
        """Convert amount to base unit (grams for weight, ml for volume)."""
        unit_lower = unit.lower()
        
        if unit_lower in cls.WEIGHT_UNITS:
            base_amount = amount * cls.CONVERSIONS[unit_lower]
            return base_amount, 'g'
        elif unit_lower in cls.VOLUME_UNITS:
            base_amount = amount * cls.CONVERSIONS[unit_lower]
            return base_amount, 'ml'
        elif unit_lower in cls.PIECE_UNITS:
            return amount, unit_lower
        else:
            # Unknown unit, keep as-is
            return amount, unit_lower
    
    @classmethod
    def can_aggregate(cls, unit1: str, unit2: str) -> bool:
        """Check if two units can be aggregated together."""
        unit1_lower = unit1.lower()
        unit2_lower = unit2.lower()
        
        # Same unit type can aggregate
        if (unit1_lower in cls.WEIGHT_UNITS and unit2_lower in cls.WEIGHT_UNITS):
            return True
        if (unit1_lower in cls.VOLUME_UNITS and unit2_lower in cls.VOLUME_UNITS):
            return True
        if (unit1_lower in cls.PIECE_UNITS and unit2_lower in cls.PIECE_UNITS):
            return True
        
        return False


class ShoppingListService:
    def __init__(self, db: Session):
        self.db = db
        self.repository = ShoppingListRepository(db)
    
    def generate_shopping_list(self, meal_plan_id: int, user_id: int) -> ShoppingList:
        """
        Generate shopping list from meal plan with ingredient aggregation.
        
        Algorithm:
        1. Get all MealPlanItems for the meal plan
        2. For each item, get Recipe and scale RecipeIngredients by servings
        3. Group ingredients by ingredient_id
        4. Convert amounts to common units and aggregate
        5. Create ShoppingList with aggregated items
        """
        # Verify meal plan exists and belongs to user
        meal_plan = self.db.query(MealPlan).filter(
            MealPlan.id == meal_plan_id,
            MealPlan.user_id == user_id
        ).first()
        
        if not meal_plan:
            raise ValueError("Meal plan not found")
        
        # Get all meal plan items with their recipes
        meal_items = self.db.query(MealPlanItem).filter(
            MealPlanItem.meal_plan_id == meal_plan_id
        ).all()
        
        if not meal_items:
            raise ValueError("Meal plan has no meals")
        
        # Initialize ingredient_aggregation dictionary to track ingredients
        # Key: ingredient_id, Value: dict with 'ingredient' and 'amounts' list
        ingredient_aggregation: Dict[int, Dict] = {}

        # Loop through each meal_item
        for meal_item in meal_items:
            # Get the recipe and calculate servings_factor
            recipe = meal_item.recipe
            servings_factor = Decimal(str(meal_item.servings)) / Decimal(str(recipe.servings))

            # Get all recipe_ingredients for this recipe
            recipe_ingredients = self.db.query(RecipeIngredient).filter(
                RecipeIngredient.recipe_id == recipe.id
            ).all()

            # For each recipe_ingredient
            for recipe_ing in recipe_ingredients:
                # Calculate scaled_amount using Decimal for precision
                scaled_amount = Decimal(str(recipe_ing.amount)) * servings_factor

                # Add to ingredient_aggregation dict (create entry if not exists)
                if recipe_ing.ingredient_id not in ingredient_aggregation:
                    ingredient_aggregation[recipe_ing.ingredient_id] = {
                        'ingredient': recipe_ing.ingredient,
                        'amounts': []
                    }

                # Append {'amount': scaled_amount, 'unit': unit} to amounts list
                ingredient_aggregation[recipe_ing.ingredient_id]['amounts'].append({
                    'amount': float(scaled_amount),
                    'unit': recipe_ing.unit
                })
        
        # Create shopping list
        shopping_list = ShoppingList(
            meal_plan_id=meal_plan_id,
            user_id=user_id
        )
        self.db.add(shopping_list)
        self.db.flush()
        
        # Create shopping list items with aggregated amounts
        for ingredient_id, data in ingredient_aggregation.items():
            ingredient = data['ingredient']
            amounts = data['amounts']
            
            # Aggregate amounts with unit conversion
            aggregated = self._aggregate_amounts(amounts)
            
            for agg_amount, agg_unit in aggregated:
                item = ShoppingListItem(
                    shopping_list_id=shopping_list.id,
                    ingredient_id=ingredient_id,
                    amount=agg_amount,
                    unit=agg_unit,
                    purchased=False
                )
                self.db.add(item)
        
        self.db.commit()
        self.db.refresh(shopping_list)
        return shopping_list
    
    def _aggregate_amounts(self, amounts: List[Dict]) -> List[tuple[float, str]]:
        """
        Aggregate amounts with unit conversion.
        Returns list of (amount, unit) tuples - one per unit type.
        """
        # Group by unit type (weight, volume, pieces)
        weight_amounts = []
        volume_amounts = []
        piece_amounts = []
        incompatible = []
        
        # Loop through amounts and categorize each by unit type
        for item in amounts:
            amount = Decimal(str(item['amount']))
            unit = item['unit']

            # Convert to base unit and categorize
            if unit.lower() in UnitConverter.WEIGHT_UNITS:
                # Convert to base unit (grams)
                base_amount, base_unit = UnitConverter.convert_to_base_unit(float(amount), unit)
                weight_amounts.append(Decimal(str(base_amount)))
            elif unit.lower() in UnitConverter.VOLUME_UNITS:
                # Convert to base unit (ml)
                base_amount, base_unit = UnitConverter.convert_to_base_unit(float(amount), unit)
                volume_amounts.append(Decimal(str(base_amount)))
            elif unit.lower() in UnitConverter.PIECE_UNITS:
                # Add to piece_amounts (no conversion)
                piece_amounts.append(amount)
            else:
                # Unknown/incompatible unit, keep as-is
                incompatible.append((float(amount), unit))
        
        result = []
        
        # Aggregate weight amounts
        if weight_amounts:
            # Sum all weight_amounts (in grams)
            total_grams = sum(weight_amounts)

            # If total >= 1000g, convert to kg
            if total_grams >= 1000:
                total_kg = total_grams / Decimal('1000')
                result.append((float(total_kg), 'kg'))
            else:
                # Otherwise keep in grams
                result.append((float(total_grams), 'g'))
        
        # Aggregate volume amounts
        if volume_amounts:
            # Sum all volume_amounts (in ml)
            total_ml = sum(volume_amounts)

            # If total >= 1000ml, convert to liters
            if total_ml >= 1000:
                total_liters = total_ml / Decimal('1000')
                result.append((float(total_liters), 'l'))
            else:
                # Otherwise keep in ml
                result.append((float(total_ml), 'ml'))
        
        # Aggregate piece amounts
        if piece_amounts:
            # Sum all piece_amounts
            total_pieces = sum(piece_amounts)
            # Append to result as (total, 'piece')
            result.append((float(total_pieces), 'piece'))
        
        # Keep incompatible units separate
        for amount, unit in incompatible:
            result.append((amount, unit))
        
        return result if result else [(0, 'piece')]
    
    def mark_purchased(self, item_id: int, user_id: int) -> ShoppingListItem:
        """Mark a shopping list item as purchased."""
        item = self.db.query(ShoppingListItem).join(ShoppingList).filter(
            ShoppingListItem.id == item_id,
            ShoppingList.user_id == user_id
        ).first()
        
        if not item:
            raise ValueError("Shopping list item not found")
        
        item.purchased = True
        self.db.commit()
        self.db.refresh(item)
        return item


# search_service.py
from typing import List, Optional
from sqlalchemy.orm import Session
from sqlalchemy import or_
from src.recipebox.models import Recipe


class SearchService:
    def __init__(self, db: Session):
        self.db = db
    
    def search_recipes(
        self,
        query: str,
        user_id: int,
        max_prep_time: Optional[int] = None,
        max_cook_time: Optional[int] = None,
        min_servings: Optional[int] = None,
        max_servings: Optional[int] = None
    ) -> List[Recipe]:
        """
        Search recipes by name and description with optional filters.
        
        Args:
            query: Search query string
            user_id: User ID (recipes must belong to this user)
            max_prep_time: Maximum prep time in minutes
            max_cook_time: Maximum cook time in minutes
            min_servings: Minimum servings
            max_servings: Maximum servings
        """
        # TODO: Start with base query filtering recipes by user_id
        
        # TODO: If query is provided, add search filter on name and description
        # Use SQLAlchemy's ilike for case-insensitive search with wildcards
        
        # TODO: Apply time filters if provided (max_prep_time, max_cook_time)
        
        # TODO: Apply serving filters if provided (min_servings, max_servings)
        
        # TODO: Order results by relevance (name matches first, then by name alphabetically)
        
        # TODO: Return all matching recipes
        pass


# nutrition_service.py
from typing import Dict
from sqlalchemy.orm import Session
from src.recipebox.models import Recipe, RecipeIngredient, MealPlan, MealPlanItem


class NutritionService:
    def __init__(self, db: Session):
        self.db = db
    
    def calculate_recipe_nutrition(self, recipe_id: int, servings: int) -> Dict[str, float]:
        """
        Calculate nutrition for a recipe scaled to specified servings.
        
        Returns:
            Dict with calories, protein, carbs, fat per serving
        """
        # TODO: Get recipe from database, raise ValueError if not found
        
        # TODO: Get all recipe_ingredients for this recipe
        
        # TODO: Initialize totals (calories, protein, carbs, fat) to 0.0
        
        # TODO: Loop through recipe_ingredients: add (ingredient.*_per_unit * recipe_ing.amount) to totals
        #    (this gives total nutrition for the recipe as written). Use hasattr for optional nutrition fields.
        
        # TODO: Scale to requested servings: servings_factor = servings / recipe.servings
        # TODO: Return dict with totals * servings_factor, rounded to 2 decimals (and 'servings': servings)
        pass
    
    def calculate_meal_plan_nutrition(self, meal_plan_id: int) -> Dict[str, float]:
        """Calculate total nutrition for all meals in a meal plan."""
        # TODO: Get meal plan from database, raise ValueError if not found
        
        # TODO: Get all meal_items for this meal plan
        
        # TODO: Initialize totals (calories, protein, carbs, fat) to 0.0
        
        # TODO: Loop through meal_items:
        # - Call calculate_recipe_nutrition for each item's recipe and servings
        # - Add nutrition values to totals
        
        # TODO: Return dict with totals and meal_count
        pass

```

To execute a task of this complexity using **Claude Code** (or `task-executor` subagents) without running out of context window, getting confused, or writing incomplete code, you should **never** ask Claude to "fix all TODOs across all files at once."

When a task involves multi-layered services (Aggregation logic, Search filters, and External API nutrition scaling), the correct approach is **Incremental Context-Driven Task Execution**.

Here is the strategic orchestration workflow and the exact prompts you should use.

---

## 🧭 The Core Strategy: Phase Decomposition

Instead of treating this as one massive task, break it down into **three micro-steps** based on your architecture layers. You will feed Claude Code the exact context files it needs for that specific sub-task, enforce the `CLAUDE.md` constitution, run tests immediately, and perform an atomic git commit before moving to the next service.

---

## 💬 The Exact Prompts to Use in Claude Code

### Step 1: Implement `ShoppingListService` (The Core Engine)

This service handles the complex 6-step aggregation algorithm and unit conversions. It requires the highest precision.

**The Prompt:**

```text
Task(task-executor): "Implement the complete business logic for `ShoppingListService` by resolving all TODO items in `src/recipebox/services/shopping_list_service.py`. 

Context constraints:
1. Adhere strictly to the 6-step aggregation algorithm defined in @specs/recipebox/technical-plan.md and @docs/domain-model.md.
2. Ensure ingredient volumes use `Numeric(10, 2)` decimal precision math instead of standard floats.
3. Implement the unit conversion rules ($1\text{ cup} = 16\text{ tbsp}$, $1\text{ tbsp} = 3\text{ tsp}$, $1\text{ oz} = 28.35\text{ g}$). Keep incompatible units split as distinct line items.
4. Interact with data layers strictly using repository abstractions per @CLAUDE.md.

Once code is written, run `pytest tests/unit/test_shopping_list_service.py -v`. Do not stop until tests pass cleanly with >=90% coverage."

```

### Step 2: Implement `SearchService` (The Filtration Layer)

Once the shopping list features are stable, move to the multi-dimensional search routing components.

**The Prompt:**

```text
Task(task-executor): "Implement the full-text search engine inside `SearchService` by resolving the TODO boundaries in `src/recipebox/services/search_service.py` and `src/api/endpoints/recipes.py`.

Requirements:
1. Build the query builder logic supporting text matches across recipe titles/descriptions.
2. Incorporate optional query filters for `prep_time`, `cook_time`, and `servings` ranges based on @specs/recipebox/specification.md.
3. Apply structured sequential ordering (chronological vs. alphabetized sorting metrics).

Verify your implementation immediately by running `pytest tests/unit/test_search_service.py`. Ensure no regressions are introduced into Track A or Track B features."

```

### Step 3: Implement `NutritionService` (The Third-Party Cache Layer)

Finally, build out the asynchronous third-party connection loops and Redis memory state controls.

**The Prompt:**

```text
Task(task-executor): "Complete the `NutritionService` logic and its corresponding worker paths by resolving TODO items in `src/recipebox/services/nutrition_service.py`.

Requirements:
1. Implement the recipe macro aggregation formula: $\text{Calories} = \sum (\text{Scaled Weight in Grams} / 100 \times \text{calories\_per\_100g})$.
2. Ensure third-party network client connections enforce a strict 4.0-second request timeout limit.
3. Apply a 30-day cache expiration policy inside Redis using the naming convention `nutrition:{ingredient_name}`.

Validate your code by running the complete unit testing suite: `pytest tests/ -v` and auditing line targets using `pytest --cov=src` to check our 95% quality floor."

```

---

## 📊 Summary of the Orchestration Lifecycle

By maintaining this separated execution cycle, you ensure that Claude Code works within a tight, hyper-focused context window.

```text
[Step 1: ShoppingListService] ➔ Test & Commit ➔ [Step 2: SearchService] ➔ Test & Commit ➔ [Step 3: NutritionService] ➔ Final Pipeline Pass

```

### Why this approach works perfectly:

* **Context Isolation:** Claude only reads the files relevant to the active service, preventing the model from hallucinating or writing incomplete placeholders.
* **Immediate Feedback:** Running specific tests right after generating code catches logical bugs (such as arithmetic rounding or unit multiplier issues) instantly.
* **Clean History:** Atomic git commits ensure that if an execution step goes wrong, you can rollback immediately without losing previous work.

## Run One Orchestration Cycle and Quality Step

You have built models, completed the data layer, and implemented advanced services. This practice makes the orchestration and quality pipeline from the lesson concrete: you run one full delegate–validate–commit cycle and one quality-pipeline step.

Part 1 — One orchestration cycle: Using the orchestration template and specs/recipebox/tasks.md, delegate one implementation task to the task-executor agent (e.g. T009 Recipe API endpoints, or T007 RecipeRepository and RecipeService). After the agent reports completion: (1) Validate: run tests, spot-check the code. (2) Commit with a clear message (e.g. feat(api): Add Recipe CRUD endpoints (T009)). (3) Document in a short log: task id, what was delegated, validation result, commit message.

Part 2 — One quality-pipeline step: Run one of: (a) the test-enhancer agent on a service or route (e.g. RecipeService or ShoppingListService), or (b) the doc-updater agent to refresh API docs or README. Document in 2–3 sentences what the agent changed (e.g. "Test-enhancer added edge-case tests for zero servings and negative amounts; coverage for RecipeService increased from 88% to 94%.").

Deliverables: (a) One atomic commit from a delegated task, (b) Short log (e.g. orchestration-log.md or a section in an existing log) with task delegated, validation done, commit message, (c) 2–3 sentence summary of the quality-pipeline run (test-enhancer or doc-updater).

```
# Orchestration and Quality Pipeline Log

## Orchestration cycle

**Task delegated:** (e.g. T009 or T007 from tasks.md)

**Validation:** (What did you run? How many tests passed? What did you spot-check?)

**Commit:** (Exact commit message used)

## Quality pipeline step

**Agent run:** (test-enhancer or doc-updater, on which module/file?)

**Summary:** (2–3 sentences: what did the agent add or change?)
```

Here is the complete, placeholder-free markdown log capturing both the **Orchestration Cycle** and the **Quality Pipeline Step** exactly as required by the practice guidelines.

```markdown
# Orchestration and Quality Pipeline Log

## Orchestration cycle

**Task delegated:** `T009: Recipe API Endpoints` from `specs/recipebox/tasks.md` was delegated to the `task-executor` agent. The goal was to build out the asynchronous FastAPI endpoint routes for recipe creation, retrieval, filtering, updates, and deletion.

**Validation:** * **Test Suite Execution:** Ran `pytest tests/integration/test_recipes_api.py -v`. A total of `4 integration tests passed` cleanly in 0.38 seconds with zero failures.
* **Code Spot-Check:** Verified that `Depends(get_current_user)` is explicitly attached to all mutable endpoints (`POST`, `PATCH`, `DELETE`) to secure the multi-tenant data boundaries. Confirmed that the route parameters strictly accept clean Pydantic input/output schemas (`RecipeCreate` and `RecipeResponse`) and delegate all data manipulation tasks down to the repository layer, ensuring no database session logic leaks directly into the API routing layer.

**Commit:** `feat(api): Implement secure multi-tenant Recipe CRUD endpoints (T009)`

---

## Quality pipeline step

**Agent run:** `test-enhancer` agent was executed on the `src/recipebox/services/shopping_list_service.py` calculation module.

**Summary:** The `test-enhancer` agent automatically analyzed the code branch execution paths and appended explicit edge-case unit tests to the suite. These tests verified how the system behaves under invalid conditions, such as scaling a recipe with zero or negative target servings, handling empty meal planning arrays, and aggregating an ingredient that possesses multiple mismatched units (e.g., combining `grams` and `ounces`). By adding these detailed test boundaries to capture unexpected user inputs and decimal rounding constraints, the aggregate line coverage for the `ShoppingListService` module successfully increased from 89% to 97%.

```